# 03 — Gold enrichment (CSV inbox → tabela UC → join)

Este notebook:
- Lê o CSV de enrichment do **container gold** (inbox)
- Publica como tabela `kabum_project.gold.enrichment_raw`
- Faz join com `kabum_project.silver.products_clean`
- Grava `kabum_project.gold.notebooks_features_enriched`

In [0]:
%run ./00_config_uc


# 00 — Config (Unity Catalog + ADLS)

Centraliza catálogo/schema, storage account e paths base.


In [0]:
spark.sql("USE CATALOG projeto_data_engineering")
spark.sql("USE SCHEMA kabum_project")

STORAGE_ACCOUNT = "storagekabumdata"

# CSV do segundo scrape para enriquecer as tabelas
GOLD_ENRICH_INBOX_DIR = f"abfss://gold@{STORAGE_ACCOUNT}.dfs.core.windows.net/inbox/kabum/gold_enrichment_v1/"

ENRICH_RAW_TABLE    = "projeto_data_engineering.kabum_project.enrichment_raw"
SILVER_TABLE        = "projeto_data_engineering.kabum_project.products_clean"
GOLD_ENRICHED_TABLE = "projeto_data_engineering.kabum_project.notebooks_features_enriched"

In [0]:
spark.sql(f"DROP TABLE IF EXISTS {ENRICH_RAW_TABLE}")

In [0]:
from pyspark.sql import functions as F

# =========================================================
# 1) Ingest CSV enrichment -> tabela (RAW)
#    - Código lê tudo como STRING para evitar drift de schema CSV tem muita variação
#    - A conversão para tipos numéricos acontece no join
# =========================================================
df_enrich_raw = (
    spark.read
      .option("header", True)
      .option("inferSchema", False)   # <- força string
      .csv(GOLD_ENRICH_INBOX_DIR)
)

# Normalização dos nomes e garantia das colunas esperadas como string
for c in ["refresh_rate_hz","brightness_nits","price","old_price","screen_resolution","panel_type","panel_finish","scrape_error","product_url"]:
    if c in df_enrich_raw.columns:
        df_enrich_raw = df_enrich_raw.withColumn(c, F.col(c).cast("string"))

(df_enrich_raw.write
  .format("delta")
  .mode("overwrite")
  .option("overwriteSchema", "true")
  .saveAsTable(ENRICH_RAW_TABLE)
)

print("OK ->", ENRICH_RAW_TABLE, "rows:", df_enrich_raw.count())


In [0]:
from pyspark.sql import functions as F

# =========================================================
# 2) Join Silver + Enrichment 
#    - converter colunas _scraped para tipos estáveis ANTES do join/write para evitar DELTA_FAILED_TO_MERGE_FIELDS)
#    - sobrescrever somente as partições do(s) ingestion_date presentes (replaceWhere)
# =========================================================

df_silver = spark.read.table(SILVER_TABLE)
df_enrich = spark.read.table(ENRICH_RAW_TABLE)

# ---------------------------------------------------------
# 2.1) Garantir ingestion_date (e tipo date) em ambos
# ---------------------------------------------------------
def ensure_ingestion_date(df):
    if "ingestion_date" not in df.columns:
        return df.withColumn("ingestion_date", F.current_date())
    return df.withColumn("ingestion_date", F.to_date(F.col("ingestion_date").cast("string")))

df_silver = ensure_ingestion_date(df_silver)
df_enrich = ensure_ingestion_date(df_enrich)

# ---------------------------------------------------------
# 2.2) Definindo a chave de join (prioridade: product_key -> product_url -> product_id)
# ---------------------------------------------------------
join_key = None
for k in ["product_key", "product_url", "product_id"]:
    if k in df_silver.columns and k in df_enrich.columns:
        join_key = k
        break

if join_key is None:
    raise Exception(
        "Não encontrei uma chave comum entre Silver e Enrichment. "
        "Tente garantir que ambos tenham 'product_key' ou 'product_url'."
    )

print("Join key:", join_key)

# ---------------------------------------------------------
# 2.3) Evita colisão de colunas: renomeia as do enrichment com sufixo _scraped (exceto a chave)
# ---------------------------------------------------------
enrich_cols = [c for c in df_enrich.columns if c != join_key]

df_enrich_ren = df_enrich.select(
    F.col(join_key),
    *[F.col(c).alias(f"{c}_scraped") for c in enrich_cols]
)

# ---------------------------------------------------------
# 2.4) Normalizações de tipos do enrichment (_scraped)
#      - A camada GOLD já tem refresh_rate_hz_scraped e brightness_nits_scraped como DOUBLE
#      - Então forcei um DOUBLE aqui (para extrair números de strings tipo "120Hz", "300 nits", etc.)
# ---------------------------------------------------------
def to_double_num(colname: str):
    s = F.col(colname).cast("string")
    s = F.regexp_replace(s, r"[^0-9,\.]", "") 
    s = F.regexp_replace(s, r",", ".")          
    return F.when(F.length(s) == 0, F.lit(None)).otherwise(s.cast("double"))

for c in ["refresh_rate_hz_scraped", "brightness_nits_scraped", "price_scraped", "old_price_scraped"]:
    if c in df_enrich_ren.columns:
        df_enrich_ren = df_enrich_ren.withColumn(c, to_double_num(c))

# strings garantidas
for c in ["screen_resolution_scraped","panel_type_scraped","panel_finish_scraped","scrape_error_scraped","product_url_scraped"]:
    if c in df_enrich_ren.columns:
        df_enrich_ren = df_enrich_ren.withColumn(c, F.col(c).cast("string"))

# boolean garantido (às vezes vem como string)
if "scrape_ok_scraped" in df_enrich_ren.columns:
    df_enrich_ren = df_enrich_ren.withColumn(
        "scrape_ok_scraped",
        F.when(F.col("scrape_ok_scraped").cast("string").isin("true", "True", "1"), F.lit(True))
         .when(F.col("scrape_ok_scraped").cast("string").isin("false", "False", "0"), F.lit(False))
         .otherwise(F.col("scrape_ok_scraped").cast("boolean"))
    )

# ---------------------------------------------------------
# 2.5) Join
# ---------------------------------------------------------
df_join = df_silver.join(df_enrich_ren, on=join_key, how="left")

# ---------------------------------------------------------
# 2.6) Coalesce: se existir a mesma coluna sem sufixo no Silver, preenche com o scraped
# ---------------------------------------------------------
for c in enrich_cols:
    sc = c
    ec = f"{c}_scraped"
    if sc in df_join.columns and ec in df_join.columns:
        df_join = df_join.withColumn(sc, F.coalesce(F.col(sc), F.col(ec)))

df_join = df_join.withColumn(
    "enriched_from_scrape",
    F.col(enrich_cols[0] + "_scraped").isNotNull() if enrich_cols else F.lit(False)
)

# =========================================================
# 3) Write Gold Enriched
#    - 1ª vez: crio uma tabela particionada (ingestion_date, search_term se existir)
#    - nas outras execuções: overwrite SOMENTE as ingestion_date presentes (replaceWhere)
# =========================================================
table_exists = spark.catalog.tableExists(GOLD_ENRICHED_TABLE)

part_cols = []
if "ingestion_date" in df_join.columns:
    part_cols.append("ingestion_date")
if "search_term" in df_join.columns:
    part_cols.append("search_term")

base_writer = (
    df_join.write
      .format("delta")
      .option("mergeSchema", "true")
      .option("overwriteSchema", "true")
)

# datas presentes no lote
ing_dates = []
if "ingestion_date" in df_join.columns:
    ing_dates = [r["ingestion_date"] for r in df_join.select("ingestion_date").distinct().collect() if r["ingestion_date"] is not None]

if not table_exists:
    writer = base_writer.mode("overwrite")
    if part_cols:
        writer = writer.partitionBy(*part_cols)
    writer.saveAsTable(GOLD_ENRICHED_TABLE)
    print(f"✅ Criada Gold: {GOLD_ENRICHED_TABLE} | rows={df_join.count()}")
else:
    # Caso não tenha ingestion_date, faz overwrite total
    if not ing_dates:
        (base_writer.mode("overwrite")
          .saveAsTable(GOLD_ENRICHED_TABLE))
        print(f"✅ Overwrite total (sem ingestion_date) em: {GOLD_ENRICHED_TABLE} | rows={df_join.count()}")
    else:
        for d in ing_dates:
            d_str = str(d) 
            df_part = df_join.filter(F.col("ingestion_date") == F.lit(d))
            (df_part.write.format("delta")
              .mode("overwrite")
              .option("mergeSchema", "true")
              .option("overwriteSchema", "true")
              .option("replaceWhere", f"ingestion_date = '{d_str}'")
              .saveAsTable(GOLD_ENRICHED_TABLE))
            print(f"✅ Overwrite por ingestion_date={d_str} em: {GOLD_ENRICHED_TABLE} | rows={df_part.count()}")


In [0]:
display(spark.read.table(GOLD_ENRICHED_TABLE).limit(30))